In [ ]:
from glob import glob
import holoviews as hv
import geoviews as gv


import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import colors as pltcolors
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Ellipse, Rectangle
from scipy.stats import linregress, chi2, ttest_ind
from cartopy import crs as ccrs, feature as cfeat
import cmweather
from datetime import datetime as dt
from metpy.plots import USCOUNTIES
plt.style.use('ams.mplstyle')
gv.extension('bokeh')

In [ ]:
all_tracks = xr.open_dataset('/Volumes/LtgSSD/tobac_saves/all_tracks.zarr', engine='zarr')

all_tracks['track_dist_from_radar'] = (all_tracks['track_projection_x_coordinate']**2 + all_tracks['track_projection_y_coordinate']**2)**0.5

close_to_khgx_mask = (all_tracks['track_dist_from_radar'].min(dim='timestep', skipna=True)/1000 <= 90)
all_tracks = all_tracks.isel(track=close_to_khgx_mask)
all_tracks['track_area'] = all_tracks['track_area'] * (0.25)**2  # convert to km^2
isolated_convective_mask = (all_tracks['track_area'].max(dim='timestep') < 450)
all_tracks = all_tracks.isel(track=isolated_convective_mask)

all_tracks['track_day'] = all_tracks.time.min(dim='timestep', skipna=True).astype('datetime64[D]')
all_tracks['track_echotop'] = all_tracks['track_echotop'] / 1000  # convert to km
all_tracks['time_since_midnight'] = (all_tracks['time'] - all_tracks['time'].data.astype('datetime64[D]')).astype(float)/1e9/3600
all_tracks['time_since_midnight'].data[all_tracks['time_since_midnight'].data < 0] = np.nan
all_tracks['track_ll_rh'] = all_tracks['track_ll_rh'] * 100  # convert to percent
all_tracks['track_sfc_rh'] = all_tracks['track_sfc_rh'] * 100  # convert to percent
all_tracks['track_duration'] = (all_tracks.time - all_tracks.time.isel(timestep=0)).astype(float)/1e9/60
all_tracks['track_duration'].data[all_tracks['track_duration'] < 0] = np.nan
all_tracks['track_seabreeze_proximity'] = all_tracks['track_seabreeze_proximity'] / 1000  # convert to km
all_tracks = all_tracks.assign({
    'track_time_gradient' : (('track', 'timestep'),  np.gradient(all_tracks.track_duration, axis=1))
})
near_seabreeze_mask = (all_tracks.track_seabreeze_proximity < 10).data
near_seabreeze_time = all_tracks.track_time_gradient.data.copy()
near_seabreeze_time[~near_seabreeze_mask] = 0
near_seabreeze_time = np.nansum(near_seabreeze_time, axis=1)
all_tracks = all_tracks.assign({
    'near_seabreeze_time' : (('track',), near_seabreeze_time)
})

flash_count = all_tracks.track_flash_count.sum(dim='timestep', skipna=True).data
flash_count[flash_count == 0] = 1e-9

In [ ]:
all_tracks['track_day'].attrs['long_name'] = 'Day of track'
all_tracks['track_day'].attrs['units'] = 'YYYY-MM-DD'

all_tracks['track_duration'].attrs['long_name'] = 'Track duration'
all_tracks['track_duration'].attrs['units'] = 'minutes'

# all_tracks['time'].attrs['long_name'] = 'Start time of radar volume scan'
all_tracks['time_since_midnight'].attrs['long_name'] = 'Start time of radar volume scan'
all_tracks['time_since_midnight'].attrs['units'] = 'UTC hour'

all_tracks['track_area'].attrs['long_name'] = 'Track footprint area'
all_tracks['track_area'].attrs['units'] = r'km$^2$'

all_tracks['track_ccl'].attrs['long_name'] = 'Cloud Condensation Level (CCL)'
all_tracks['track_ccl'].attrs['units'] = 'hPa'

all_tracks['track_ccn_profile_0.4'].attrs['long_name'] = 'Surface CCN concentration at 0.4% supersaturation'
all_tracks['track_ccn_profile_0.4'].attrs['units'] = 'cm$^{-3}$'


all_tracks['track_ccn_profile_0.6'].attrs['long_name'] = 'Surface CCN concentration at 0.6% supersaturation'
all_tracks['track_ccn_profile_0.6'].attrs['units'] = 'cm$^{-3}$'

all_tracks['track_child_cell_count'].attrs['long_name'] = 'Number of child cells in track'
all_tracks['track_child_cell_count'].attrs['units'] = 'count'

all_tracks['track_convT'].attrs['long_name'] = 'Convective Temperature'
all_tracks['track_convT'].attrs['units'] = '°C'

all_tracks['track_dewpoint_profile'].attrs['long_name'] = 'Surface dewpoint temperature'
all_tracks['track_dewpoint_profile'].attrs['units'] = '°C'

all_tracks['track_echotop'].attrs['long_name'] = '18 dBZ echo top height'
all_tracks['track_echotop'].attrs['units'] = 'km'

all_tracks['track_el'].attrs['long_name'] = 'Track Equilibrium Level (EL)'
all_tracks['track_el'].attrs['units'] = 'hPa'

all_tracks['track_kdpcol'].attrs['long_name'] = r'K$_{DP}$ values > 0.75 °/km, column summed, maximum over feature footprint'
all_tracks['track_kdpcol'].attrs['units'] = '°/km'

all_tracks['track_kdpcol_mean'].attrs['long_name'] = r'K$_{DP}$ values > 0.75 °/km, column summed, averaged over feature footprint'
all_tracks['track_kdpcol_mean'].attrs['units'] = '°/km'

all_tracks['track_kdpcol_total'].attrs['long_name'] = r'K$_{DP}$ values > 0.75 °/km, summed over feature volume'
all_tracks['track_kdpcol_total'].attrs['units'] = '°/km'

all_tracks['track_kdpvol'].attrs['long_name'] = r'Tobac grid cells with K$_{DP}$ > 0.75 °/km in feature footprint'
all_tracks['track_kdpvol'].attrs['units'] = 'count'

all_tracks['track_kdpwt_total'].attrs['long_name'] = r'K$_{DP}$ values > 0.75 °/km, weighted by height above the 0$^\circ$C level,'+'\ncolumn summed, total over feature footprint'
all_tracks['track_kdpwt_total'].attrs['units'] = 'm°/km'

all_tracks['track_lat'].attrs['long_name'] = 'Latitude of track centroid'
all_tracks['track_lat'].attrs['units'] = '°N'

all_tracks['track_lon'].attrs['long_name'] = 'Longitude of track centroid'
all_tracks['track_lon'].attrs['units'] = '°E'

all_tracks['track_lcl'].attrs['long_name'] = 'Lifted Condensation Level (LCL)'
all_tracks['track_lcl'].attrs['units'] = 'hPa'

all_tracks['track_lfc'].attrs['long_name'] = 'Level of Free Convection (LFC)'
all_tracks['track_lfc'].attrs['units'] = 'hPa'

all_tracks['track_max_reflectivity'].attrs['long_name'] = 'Maximum reflectivity in feature volume'
all_tracks['track_max_reflectivity'].attrs['units'] = 'dBZ'

all_tracks['track_ll_rh'].attrs['long_name'] = '0-6 km mean relative humidity'
all_tracks['track_ll_rh'].attrs['units'] = '%' 

all_tracks['track_min_L2_MCMIPC'].attrs['long_name'] = 'Minimum channel 13 brightness temperature in feature footprint'
all_tracks['track_min_L2_MCMIPC'].attrs['units'] = 'K'

all_tracks['track_mlcape'].attrs['long_name'] = '100 hPa Mixed-layer CAPE'
all_tracks['track_mlcape'].attrs['units'] = 'J/kg'

all_tracks['track_mlcin'].attrs['long_name'] = '100 hPa Mixed-layer CINH'
all_tracks['track_mlcin'].attrs['units'] = 'J/kg'

all_tracks['track_mlecape'].attrs['long_name'] = '100 hPa Mixed-layer Entrainment CAPE'
all_tracks['track_mlecape'].attrs['units'] = 'J/kg'

all_tracks['track_pressure_profile'].attrs['long_name'] = 'Surface pressure'
all_tracks['track_pressure_profile'].attrs['units'] = 'hPa'

all_tracks['track_rhvdeficitcol'].attrs['long_name'] = r'$\rho_{HV}$ deficit values > 0.02, column summed, maximum over feature footprint'
all_tracks['track_rhvdeficitcol'].attrs['units'] = 'unitless'

all_tracks['track_rhvdeficitcol_mean'].attrs['long_name'] = r'$\rho_{HV}$ deficit values > 0.02, column summed, averaged over feature footprint'
all_tracks['track_rhvdeficitcol_mean'].attrs['units'] = 'unitless'

all_tracks['track_rhvdeficitcol_total'].attrs['long_name'] = r'$\rho_{HV}$ deficit values > 0.02, summed over feature volume'
all_tracks['track_rhvdeficitcol_total'].attrs['units'] = 'unitless'

all_tracks['track_rhvdeficitvol'].attrs['long_name'] = r'Tobac grid cells with $\rho_{HV}$ deficit values > 0.02 in feature footprint'
all_tracks['track_rhvdeficitvol'].attrs['units'] = 'count'

all_tracks['track_rhvdeficitwt_total'].attrs['long_name'] = r'$\rho_{HV}$ deficit values > 0.02, weighted by height above the 0$^\circ$C level,'+'\ncolumn summed, total over feature footprint'
all_tracks['track_rhvdeficitwt_total'].attrs['units'] = 'm'

all_tracks['track_seabreeze'].attrs['long_name'] = 'Sea breeze flag'
all_tracks['track_seabreeze'].attrs['units'] = '-2 = continental, -1 = maritime'

all_tracks['track_seabreeze_proximity'].attrs['long_name'] = 'Distance from analyzed sea breeze front'
all_tracks['track_seabreeze_proximity'].attrs['units'] = 'km'

all_tracks['track_sfc_rh'].attrs['long_name'] = 'Surface relative humidity'
all_tracks['track_sfc_rh'].attrs['units'] = '%'

all_tracks['track_six_km_bwd'].attrs['long_name'] = 'Surface to 6 km bulk wind difference'
all_tracks['track_six_km_bwd'].attrs['units'] = 'm/s'

all_tracks['track_six_km_lapse'].attrs['long_name'] = 'Surface to 6 km lapse rate'
all_tracks['track_six_km_lapse'].attrs['units'] = 'K/km'

all_tracks['track_temperature_profile'].attrs['long_name'] = 'Surface temperature'
all_tracks['track_temperature_profile'].attrs['units'] = '°C'

all_tracks['track_zdrcol'].attrs['long_name'] = r'Z$_{DR}$ values > 1 dB, column summed, maximum over track footprint'
all_tracks['track_zdrcol'].attrs['units'] = 'dB'

all_tracks['track_zdrcol_mean'].attrs['long_name'] = r'Z$_{DR}$ values > 1 dB, column summed, averaged over track footprint'
all_tracks['track_zdrcol_mean'].attrs['units'] = 'dB'

all_tracks['track_zdrcol_total'].attrs['long_name'] = r'Z$_{DR}$ values > 1 dB, column summed, total over track footprint'
all_tracks['track_zdrcol_total'].attrs['units'] = 'dB'

all_tracks['track_zdrvol'].attrs['long_name'] = r'Tobac grid cells with Z$_{DR}$ > 1 dB in track footprint'
all_tracks['track_zdrvol'].attrs['units'] = 'count'

all_tracks['track_zdrwt_total'].attrs['long_name'] = r'Z$_{DR}$ values > 1 dB, weighted by height above the 0$^\circ$C level,\ncolumn summed, total over track footprint'
all_tracks['track_zdrwt_total'].attrs['units'] = 'm dB'

all_tracks['near_seabreeze_time'].attrs['long_name'] = 'Time within 10 km of sea breeze front'
all_tracks['near_seabreeze_time'].attrs['units'] = 'min'

has_kdp_mask = all_tracks.track_kdpcol.sum(dim='timestep') > 0
has_zdr_mask = all_tracks.track_zdrcol.sum(dim='timestep').data > 0
has_ltg_mask = all_tracks.track_flash_count.sum(dim='timestep').data > 0

nothing_tracks = all_tracks.isel(track=((~has_kdp_mask) & (~has_zdr_mask) & (~has_ltg_mask)))
zdr_tracks = all_tracks.isel(track=(has_zdr_mask & (~has_kdp_mask) & (~has_ltg_mask)))
kdp_tracks = all_tracks.isel(track=((~has_zdr_mask) & (has_kdp_mask) & (~has_ltg_mask)))
zdr_kdp_tracks = all_tracks.isel(track=(has_zdr_mask & has_kdp_mask & (~has_ltg_mask)))
zdr_ltg_tracks = all_tracks.isel(track=(has_zdr_mask & (~has_kdp_mask) & has_ltg_mask))
kdp_ltg_tracks = all_tracks.isel(track=((~has_zdr_mask) & has_kdp_mask & has_ltg_mask))
zdr_kdp_ltg_tracks = all_tracks.isel(track=(has_zdr_mask & has_kdp_mask & has_ltg_mask))
ltg_tracks = all_tracks.isel(track=((~has_zdr_mask) & (~has_kdp_mask) & has_ltg_mask))

In [ ]:
normalized_times = xr.open_dataset('/Volumes/LtgSSD/tobac_saves/regular_time_tracks.zarr', engine='zarr')
close_to_khgx_mask = (((normalized_times.track_projection_x_coordinate ** 2 + normalized_times.track_projection_y_coordinate ** 2)**0.5) / 1000).min(dim='time', skipna=True) <= 90
normalized_times = normalized_times.isel(track=close_to_khgx_mask)
normalized_times['track_area'] = normalized_times['track_area'] * (0.25)**2  # convert to km^2
isolated_convective_mask = (normalized_times['track_area'].max(dim='time') < 450)
normalized_times = normalized_times.isel(track=isolated_convective_mask)
normalized_times['track_present'] = ~np.isnan(normalized_times.track_seabreeze)

has_kdp_mask = normalized_times.track_kdpcol.sum(dim='time') > 0
has_zdr_mask = normalized_times.track_zdrcol.sum(dim='time').data > 0
has_ltg_mask = normalized_times.track_flash_count.sum(dim='time').data > 0
nothing_normalized = normalized_times.isel(track=((~has_kdp_mask) & (~has_zdr_mask) & (~has_ltg_mask)))
zdr_normalized = normalized_times.isel(track=(has_zdr_mask & (~has_kdp_mask) & (~has_ltg_mask)))
kdp_normalized = normalized_times.isel(track=((~has_zdr_mask) & (has_kdp_mask) & (~has_ltg_mask)))
zdr_kdp_normalized = normalized_times.isel(track=(has_zdr_mask & has_kdp_mask & (~has_ltg_mask)))
zdr_ltg_normalized = normalized_times.isel(track=(has_zdr_mask & (~has_kdp_mask) & has_ltg_mask))
kdp_ltg_normalized = normalized_times.isel(track=((~has_zdr_mask) & has_kdp_mask & has_ltg_mask))
zdr_kdp_ltg_normalized = normalized_times.isel(track=(has_zdr_mask & has_kdp_mask & has_ltg_mask))
ltg_normalized = normalized_times.isel(track=((~has_zdr_mask) & (~has_kdp_mask) & has_ltg_mask))

In [ ]:
def calculate_error_ellipse(x, y, confidence_interval=0.68):
    """Calculate parameters for an error ellipse representing the covariance of x and y data.
    
    Parameters
    ----------
    x : array-like
        1D array of x data points.
    y : array-like
        1D array of y data points.
    confidence_interval : float, optional
        Confidence interval for the ellipse (default is 0.68 for 68% confidence).
    
    Returns
    -------
    mu_x : float
        Mean of x data points.
    mu_y : float
        Mean of y data points.
    ell_x : np.ndarray
        Lengths of the semi-major and semi-minor axes of the ellipse.
    angle : float
        Rotation angle of the ellipse in degrees.
    """
    chi2_val = chi2.ppf(confidence_interval, 2)
    mu_x, mu_y = np.mean(x), np.mean(y)
    eig = np.linalg.eig(np.cov(x, y))
    lambda_x = eig.eigenvalues
    nu_x_2, nu_x_1 = eig.eigenvectors
    ell_x = 2*(chi2_val*lambda_x)**0.5
    angle = np.rad2deg(np.arctan2(nu_x_1[0], nu_x_1[1]))
    return mu_x, mu_y, ell_x, angle

styles_dict = {
    'nothing' : {'color' : 'tab:blue', 'label' : 'Nothing', 'alpha' : 0.2, 'linewidth' : 1},
    'zdr' : {'color' : 'tab:orange', 'label' : r'Z$_{DR}$', 'alpha' : 0.2, 'linewidth' : 1},
    'kdp' : {'color' : 'tab:green', 'label' : r'K$_{DP}$', 'alpha' : 0.2, 'linewidth' : 1},
    'zdr_kdp' : {'color' : 'tab:red', 'label' : r'Z$_{DR}$ K$_{DP}$', 'alpha' : 0.2, 'linewidth' : 1},
    'zdr_ltg' : {'color' : 'tab:purple', 'label' : r'Z$_{DR}$ Lightning', 'alpha' : 0.2, 'linewidth' : 1},
    'kdp_ltg' : {'color' : 'tab:brown', 'label' : r'K$_{DP}$ Lightning', 'alpha' : 0.2, 'linewidth' : 1},
    'zdr_kdp_ltg' : {'color' : 'tab:pink', 'label' : r'Z$_{DR}$ K$_{DP}$ Lightning', 'alpha' : 0.2, 'linewidth' : 1},
    'ltg' : {'color' : 'tab:gray', 'label' : 'Lightning Only', 'alpha' : 0.2, 'linewidth' : 1}
}
flash_counts_dict = {
    'nothing' : nothing_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'zdr' : zdr_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'kdp' : kdp_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'zdr_kdp' : zdr_kdp_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'zdr_ltg' : zdr_ltg_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'kdp_ltg' : kdp_ltg_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'zdr_kdp_ltg' : zdr_kdp_ltg_tracks.track_flash_count.sum(dim='timestep', skipna=True),
    'ltg' : ltg_tracks.track_flash_count.sum(dim='timestep', skipna=True)
}

In [ ]:
track_counts_fig = plt.figure(figsize=(6.5, 8.5))
counts_gs = GridSpec(5, 2, figure=track_counts_fig, height_ratios=[1e-2, 1, 1e-2, 1, 1e-4])
axs_counts = [track_counts_fig.add_subplot(counts_gs[i, j]) for i in [1, 3] for j in range(2)]
axs_legends = [track_counts_fig.add_subplot(counts_gs[i, j]) for i in [0, 2] for j in range(2)]
axs_counts.append(track_counts_fig.add_subplot(counts_gs[-1, :]))
for j, seabreeze_side_select2 in enumerate(['all', 'continental', 'maritime', 'crossing']):
    unique_days = np.sort(np.unique(all_tracks['track_day'].data))
    nothing_sum = np.zeros(unique_days.shape, dtype=int)
    zdr_sum = np.zeros(unique_days.shape, dtype=int)
    kdp_sum = np.zeros(unique_days.shape, dtype=int)
    zdr_kdp_sum = np.zeros(unique_days.shape, dtype=int)
    zdr_ltg_sum = np.zeros(unique_days.shape, dtype=int)
    kdp_ltg_sum = np.zeros(unique_days.shape, dtype=int)
    zdr_kdp_ltg_sum = np.zeros(unique_days.shape, dtype=int)
    ltg_sum = np.zeros(unique_days.shape, dtype=int)
    rolling_sum = np.zeros(unique_days.shape, dtype=int)
    track_counts_ax = axs_counts[j]
    if seabreeze_side_select2 == 'all':
        seabreeze_mask_nothing = np.ones(nothing_tracks['track'].size, dtype=bool)
        seabreeze_mask_zdr = np.ones(zdr_tracks['track'].size, dtype=bool)
        seabreeze_mask_kdp = np.ones(kdp_tracks['track'].size, dtype=bool)
        seabreeze_mask_zdr_kdp = np.ones(zdr_kdp_tracks['track'].size, dtype=bool)
        seabreeze_mask_zdr_ltg = np.ones(zdr_ltg_tracks['track'].size, dtype=bool)
        seabreeze_mask_kdp_ltg = np.ones(kdp_ltg_tracks['track'].size, dtype=bool)
        seabreeze_mask_zdr_kdp_ltg = np.ones(zdr_kdp_ltg_tracks['track'].size, dtype=bool)
        seabreeze_mask_ltg = np.ones(ltg_tracks['track'].size, dtype=bool)
    elif seabreeze_side_select2 == 'continental':
        seabreeze_mask_nothing = nothing_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_zdr = zdr_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_kdp = kdp_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_zdr_kdp = zdr_kdp_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_zdr_ltg = zdr_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_kdp_ltg = kdp_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_zdr_kdp_ltg = zdr_kdp_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
        seabreeze_mask_ltg = ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) == -2
    elif seabreeze_side_select2 == 'maritime':
        seabreeze_mask_nothing = nothing_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_zdr = zdr_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_kdp = kdp_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_zdr_kdp = zdr_kdp_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_zdr_ltg = zdr_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_kdp_ltg = kdp_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_zdr_kdp_ltg = zdr_kdp_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
        seabreeze_mask_ltg = ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) == -1
    elif seabreeze_side_select2 == 'crossing':
        seabreeze_mask_nothing = (nothing_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (nothing_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_zdr = (zdr_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (zdr_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_kdp = (kdp_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (kdp_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_zdr_kdp = (zdr_kdp_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (zdr_kdp_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_zdr_ltg = (zdr_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (zdr_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_kdp_ltg = (kdp_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (kdp_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_zdr_kdp_ltg = (zdr_kdp_ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (zdr_kdp_ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
        seabreeze_mask_ltg = (ltg_tracks.track_seabreeze.min(dim='timestep', skipna=True) != -1) & (ltg_tracks.track_seabreeze.max(dim='timestep', skipna=True) != -2)
    for i, day in enumerate(unique_days):
        nothing_sum[i] = (nothing_tracks.isel(track=seabreeze_mask_nothing).track_day == day).sum()
        zdr_sum[i] = (zdr_tracks.isel(track=seabreeze_mask_zdr).track_day == day).sum()
        kdp_sum[i] = (kdp_tracks.isel(track=seabreeze_mask_kdp).track_day == day).sum()
        zdr_kdp_sum[i] = (zdr_kdp_tracks.isel(track=seabreeze_mask_zdr_kdp).track_day == day).sum()
        zdr_ltg_sum[i] = (zdr_ltg_tracks.isel(track=seabreeze_mask_zdr_ltg).track_day == day).sum()
        kdp_ltg_sum[i] = (kdp_ltg_tracks.isel(track=seabreeze_mask_kdp_ltg).track_day == day).sum()
        zdr_kdp_ltg_sum[i] = (zdr_kdp_ltg_tracks.isel(track=seabreeze_mask_zdr_kdp_ltg).track_day == day).sum()
        ltg_sum[i] = (ltg_tracks.isel(track=seabreeze_mask_ltg).track_day == day).sum()
    unique_days = [d.strftime('%m/%d') for d in unique_days.astype('O')]
    nothing_bar = track_counts_ax.bar(unique_days, nothing_sum, color=styles_dict['nothing']['color'], label=styles_dict['nothing']['label'], linewidth=styles_dict['nothing']['linewidth'])
    rolling_sum += nothing_sum
    zdr_bar = track_counts_ax.bar(unique_days, zdr_sum, bottom=rolling_sum, color=styles_dict['zdr']['color'], label=styles_dict['zdr']['label'], linewidth=styles_dict['zdr']['linewidth'])
    rolling_sum += zdr_sum
    kdp_bar = track_counts_ax.bar(unique_days, kdp_sum, bottom=rolling_sum, color=styles_dict['kdp']['color'], label=styles_dict['kdp']['label'], linewidth=styles_dict['kdp']['linewidth'])
    rolling_sum += kdp_sum
    zdr_kdp_bar = track_counts_ax.bar(unique_days, zdr_kdp_sum, bottom=rolling_sum, color=styles_dict['zdr_kdp']['color'], label=styles_dict['zdr_kdp']['label'], linewidth=styles_dict['zdr_kdp']['linewidth'])
    rolling_sum += zdr_kdp_sum
    zdr_ltg_bar = track_counts_ax.bar(unique_days, zdr_ltg_sum, bottom=rolling_sum, color=styles_dict['zdr_ltg']['color'], label=styles_dict['zdr_ltg']['label'], linewidth=styles_dict['zdr_ltg']['linewidth'])
    rolling_sum += zdr_ltg_sum
    kdp_ltg_bar = track_counts_ax.bar(unique_days, kdp_ltg_sum, bottom=rolling_sum, color=styles_dict['kdp_ltg']['color'], label=styles_dict['kdp_ltg']['label'], linewidth=styles_dict['kdp_ltg']['linewidth'])
    rolling_sum += kdp_ltg_sum
    zdr_kdp_ltg_bar = track_counts_ax.bar(unique_days, zdr_kdp_ltg_sum, bottom=rolling_sum, color=styles_dict['zdr_kdp_ltg']['color'], label=styles_dict['zdr_kdp_ltg']['label'], linewidth=styles_dict['zdr_kdp_ltg']['linewidth'])
    rolling_sum += zdr_kdp_ltg_sum
    ltg_bar = track_counts_ax.bar(unique_days, ltg_sum, bottom=rolling_sum, color=styles_dict['ltg']['color'], label=styles_dict['ltg']['label'], linewidth=styles_dict['ltg']['linewidth'])
    rolling_sum += ltg_sum
    if j % 2 == 0:
        track_counts_ax.set_ylabel('Track Count')
    if j > 1:
        pass
        # track_counts_ax.set_xlabel('Day')
    track_counts_ax.xaxis.set_tick_params(rotation=90)
    if j == 0:
        axs_counts[-1].legend(handles=[nothing_bar, zdr_bar, kdp_bar, zdr_kdp_bar, zdr_ltg_bar, kdp_ltg_bar, zdr_kdp_ltg_bar, ltg_bar], loc='center', ncols=4)
        axs_counts[-1].axis('off')
    all_sums_this_side = np.array([nothing_sum.sum(), zdr_sum.sum(), kdp_sum.sum(), zdr_kdp_sum.sum(), zdr_ltg_sum.sum(),
                                   kdp_ltg_sum.sum(), zdr_kdp_ltg_sum.sum(), ltg_sum.sum()]).astype(int)
    axs_legends[j].legend(handles=[nothing_bar, zdr_bar, kdp_bar, zdr_kdp_bar, zdr_ltg_bar, kdp_ltg_bar, zdr_kdp_ltg_bar, ltg_bar],
                          loc='center', ncols=4,
                          labels=[f'{s}' for s in all_sums_this_side], title=f'Totals for {seabreeze_side_select2.title()} Tracks')
    axs_legends[j].axis('off')

top_legend_offset = .005
left_legend_offset = -.075
middle_legend_offset = -.075
bottom_counts_offset = -.005


row0_legend_y1 = 0.8767140581978701 + top_legend_offset
legend_width = 0.35227272727272724
legend_height = 0.0032859418021299325

row1_legend_y1 = 0.4917304879068806
#-------------------------------------------
col1_count_x1 = 0.17
left_count_offset = -0.1
col2_count_x1 = 0.50
row1_count_y1 = 0.5215681538469417
count_width = 0.42
count_height = 0.32859418021299736

row2_count_y1 = 0.13658458355595215

axs_legends[0].set_position([col1_count_x1+left_legend_offset, row0_legend_y1, legend_width, legend_height])
axs_legends[1].set_position([col2_count_x1-left_legend_offset, row0_legend_y1, legend_width, legend_height])
axs_counts[0].set_position([col1_count_x1+left_count_offset, row1_count_y1, count_width, count_height])
axs_counts[1].set_position([col2_count_x1-(left_count_offset/2), row1_count_y1, count_width, count_height])

axs_legends[2].set_position([col1_count_x1+left_legend_offset, row1_legend_y1+middle_legend_offset, legend_width, legend_height])
axs_legends[3].set_position([col2_count_x1-left_legend_offset, row1_legend_y1+middle_legend_offset, legend_width, legend_height])
axs_counts[2].set_position([col1_count_x1+left_count_offset, row2_count_y1+middle_legend_offset+bottom_counts_offset, count_width, count_height])
axs_counts[3].set_position([col2_count_x1-(left_count_offset/2), row2_count_y1+middle_legend_offset+bottom_counts_offset, count_width, count_height])

axs_counts[-1].set_position([0.05, -0.015, 0.9, 0])

track_counts_fig.savefig('mwr_figs/track_day_counts.pdf') # Fig. 2
plt.close(track_counts_fig)

In [ ]:
def make_combined_frequency_presence(dv, this_func, x_bins, should_xlog=False, isel=None):
    thing_to_plot = all_tracks[dv]
    if isel is not None:
        thing_to_plot = thing_to_plot.isel(**isel)
    if this_func == 'mean':
        thing_to_plot = thing_to_plot.mean(dim='timestep', skipna=True)
        fancy_string = ',\nmean along track'
    elif this_func == 'min':
        thing_to_plot = thing_to_plot.min(dim='timestep', skipna=True)
        fancy_string = ',\nminimum along track'
    elif this_func == 'max':
        thing_to_plot = thing_to_plot.max(dim='timestep', skipna=True)
        fancy_string = ',\nmaximum along track'
    elif this_func == 'sum':
        thing_to_plot = thing_to_plot.sum(dim='timestep', skipna=True)
        fancy_string = ',\nsum along track'
    elif this_func == 'nothing':
        fancy_string = ''
    else:
        raise ValueError(f"Unknown function {this_func} for variable {dv}")
    if 'time' in thing_to_plot.name:
        thing_to_plot = thing_to_plot.data.astype(float)

    x_bin0 = x_bins[0]
    x_binf = x_bins[-1]
    if should_xlog:
        thing_to_plot = thing_to_plot.copy()
        thing_to_plot[thing_to_plot == 0] = 1e-11
        x_bins = np.append([1e-9], x_bins)
    x_bins[0] = np.min([thing_to_plot.min(), x_bin0])
    x_bins[-1] = np.max([thing_to_plot.max(), x_binf])
    y_bins = np.logspace(0, 3, 100)
    y_bins = np.append([1e-8], y_bins)
    y_bin0 = y_bins[0]
    y_binf = y_bins[-1]
    y_bins[0] = np.min([flash_count.min(), y_bin0])
    y_bins[-1] = np.max([flash_count.max(), y_binf])
    fig = plt.figure(figsize=(6.5, 5))
    gs = GridSpec(5, 3, figure=fig, width_ratios=[1, 1, 1], height_ratios=[0.1, 0.9, 1, 1, 1])
    ax = plt.subplot(gs[1:3, 0])
    art = ax.hist2d(
        thing_to_plot,
        flash_count,
        cmap='viridis', norm=pltcolors.LogNorm(), bins=[x_bins, y_bins], range=[[thing_to_plot.min(), thing_to_plot.max()], [flash_count.min(), flash_count.max()]], rasterized=True,
    )
    thing_to_plot_nonnan = thing_to_plot[~np.isnan(thing_to_plot) & ~np.isnan(flash_count)]
    flash_count_nonnan = flash_count[~np.isnan(thing_to_plot) & ~np.isnan(flash_count)]
    ax.set_yscale('log')
    
    if should_xlog:
        ax.set_xscale('log')
        x_bin0 -= 10**int((np.log10(x_bin0) - 1))
    ax.set_xlim(x_bin0, x_binf)
    ax.set_ylim(0.9, y_binf)
    # ax.set_xlabel(f'{all_tracks[dv].attrs.get("long_name", dv)}{fancy_string} ({all_tracks[dv].attrs.get("units", "")})')
    ax.set_ylabel('Track Flash Count')
    cbar_ax = plt.subplot(gs[0, 0])
    cbar = fig.colorbar(art[3], ax=ax, cax=cbar_ax, orientation='horizontal', label='Track count')
    cbar_ax.xaxis.set_ticks_position('top')
    mu_x, mu_y, ell_x, angle = calculate_error_ellipse(thing_to_plot_nonnan, flash_count_nonnan)
    ellipse = Ellipse((mu_x, mu_y), width=ell_x[0], height=ell_x[1], angle=angle,
                      fill=False, color='k', linewidth=1, alpha=0.5, label=r'1-$\sigma$ (all)')
    ax.add_patch(ellipse)
    ax.scatter(mu_x, mu_y, color='k', marker='P', s=20, linewidths=0.75, edgecolors='white')
    mu_x_ltg, mu_y_ltg, ell_x_ltg, angle_ltg = calculate_error_ellipse(thing_to_plot_nonnan[flash_count_nonnan >= 1], flash_count_nonnan[flash_count_nonnan >= 1])
    ellipse_ltg = Ellipse((mu_x_ltg, mu_y_ltg), width=ell_x_ltg[0], height=ell_x_ltg[1], angle=angle_ltg,
                          fill=False, color='r', linewidth=1, alpha=0.5, label=r'1-$\sigma$ (lightning > 0)')
    ax.add_patch(ellipse_ltg)
    ax.scatter(mu_x_ltg, mu_y_ltg, color='r', marker='P', s=20, linewidths=0.75, edgecolors='white')
    lax1 = plt.subplot(gs[3, :])
    lax1.axis('off')
    try:
        reg_base = np.linspace(x_bin0, x_binf, 1000)
        this_reg = linregress(thing_to_plot_nonnan, flash_count_nonnan)
        this_pval_str = f'={this_reg.pvalue:.3f}' if this_reg.pvalue >= 0.001 else f'<.001'
        all_reg_handle = ax.plot(reg_base, this_reg.intercept + this_reg.slope*reg_base, color='k', linewidth=1, linestyle='--', label=f'y={this_reg.slope:.2f}x+{this_reg.intercept:.1f}\n'+r'r$^{2}$'+f'={this_reg.rvalue**2:.2f}, p{this_pval_str}')
        ltg_reg = linregress(thing_to_plot_nonnan[flash_count_nonnan >= 1], flash_count_nonnan[flash_count_nonnan >= 1])
        ltg_pval_str = f'={ltg_reg.pvalue:.3f}' if ltg_reg.pvalue >= 0.001 else f'<.001'
        ltg_reg_handle = ax.plot(reg_base, ltg_reg.intercept + ltg_reg.slope*reg_base, color='r', linewidth=1, linestyle='--', label=f'y={ltg_reg.slope:.2f}x+{ltg_reg.intercept:.1f}\n'+r'r$^{2}$'+f'={ltg_reg.rvalue**2:.2f}, p{ltg_pval_str}')
    except ValueError:
        print(f"Skipping regression for {dv} with {this_func} due to insufficient data.")
    track_count_ax = plt.subplot(gs[0:2, 1])
    track_percent_ax = plt.subplot(gs[0:2, 2])
    ltg_reg_ax = plt.subplot(gs[2, 1])
    cat_reg_ax = plt.subplot(gs[2, 2])
    legend_ax = plt.subplot(gs[-1, :])
    axs = [track_count_ax, track_percent_ax, cat_reg_ax, ltg_reg_ax, legend_ax]

    things_to_plot = {
        'nothing' : nothing_tracks[dv],
        'zdr' : zdr_tracks[dv],
        'kdp' : kdp_tracks[dv],
        'zdr_kdp' : zdr_kdp_tracks[dv],
        'zdr_ltg' : zdr_ltg_tracks[dv],
        'kdp_ltg' : kdp_ltg_tracks[dv],
        'zdr_kdp_ltg' : zdr_kdp_ltg_tracks[dv],
        'ltg' : ltg_tracks[dv],
    }
    if isel is not None:
        for key, val in things_to_plot.items():
            things_to_plot[key] = val.isel(**isel)
    if this_func == 'mean':
        for key, val in things_to_plot.items():
            things_to_plot[key] = val.mean(dim='timestep', skipna=True)
    elif this_func == 'min':
        for key, val in things_to_plot.items():
            things_to_plot[key] = val.min(dim='timestep', skipna=True)
    elif this_func == 'max':
        for key, val in things_to_plot.items():
            things_to_plot[key] = val.max(dim='timestep', skipna=True)
    elif this_func == 'sum':
        for key, val in things_to_plot.items():
            things_to_plot[key] = val.sum(dim='timestep', skipna=True)
    elif this_func == 'nothing':
        fancy_string = ''
    else:
        raise ValueError(f"Unknown function {this_func} for variable {dv}")
    x_bins[0] = np.min([np.nanmin(np.hstack(list(things_to_plot.values()))), x_bin0])
    x_bins[-1] = np.max([np.nanmax(np.hstack(list(things_to_plot.values()))), x_binf])
    axs[0].hist(things_to_plot.values(), histtype='barstacked', label=[styles_dict[key]['label'] for key in things_to_plot.keys()],
            color=[styles_dict[key]['color'] for key in things_to_plot.keys()], bins=x_bins)
    histos = np.array([np.histogram(t, bins=x_bins)[0] for t in things_to_plot.values()])
    histo_fracs = (histos/np.sum(histos, axis=0))*100
    histo_fracs[np.isnan(histo_fracs)] = 0
    hist_handles = [axs[1].stairs(np.sum(histo_fracs[0:i+1], axis=0), x_bins, baseline=np.sum(histo_fracs[0:i], axis=0), label=styles_dict[key]['label'], color=styles_dict[key]['color'], linewidth=1, fill=True) for i, key in enumerate(things_to_plot.keys())]
    x_bin_ctr = (x_bins[:-1] + x_bins[1:])/2

    invalid_histo_mask = np.sum(histos, axis=0) > 30
    if dv == 'track_seabreeze_proximity':
        invalid_histo_mask = x_bin_ctr <= 20
        histo_fracs_lr = histo_fracs[:, ~invalid_histo_mask]
        x_bin_ctr_lr = x_bin_ctr.copy()[~invalid_histo_mask]
        [axs[2].scatter(x_bin_ctr_lr, histo_fracs_lr[i], color=styles_dict[key]['color'], s=2, alpha=0.2) for i, key in enumerate(things_to_plot.keys())]
        regrs_lr = [linregress(x_bin_ctr_lr, histo_fracs_lr[i]) for i in range(histo_fracs_lr.shape[0])]
        [axs[2].plot(x_bin_ctr_lr, regrs_lr[i].intercept + regrs_lr[i].slope*x_bin_ctr_lr, color=styles_dict[key]['color'], linestyle='--', linewidth=1, alpha=0.2) for i, key in enumerate(things_to_plot.keys())]
        nonltg_lr = np.zeros(histo_fracs_lr.shape[1], dtype=float)
        ltg_lr = np.zeros(histo_fracs_lr.shape[1], dtype=float)
        for i, histo_frac in enumerate(histo_fracs_lr):
            if 'Lightning' in styles_dict[list(things_to_plot.keys())[i]]['label']:
                ltg_lr += histo_frac
            else:
                nonltg_lr += histo_frac
        invalid_mask_lr = (nonltg_lr + ltg_lr) == 0
        x_bin_ctr_lr = x_bin_ctr_lr[~invalid_mask_lr]
        nonltg_lr = nonltg_lr[~invalid_mask_lr]
        ltg_lr = ltg_lr[~invalid_mask_lr]
        nonltg_pts_lr = axs[3].scatter(x_bin_ctr_lr, nonltg_lr, color='k', s=2, alpha=0.2)
        ltg_pts_lr = axs[3].scatter(x_bin_ctr_lr, ltg_lr, color='tab:olive', s=2, alpha=0.2)
        regr_lr_nonltg = linregress(x_bin_ctr_lr, nonltg_lr)
        regr_lr_ltg = linregress(x_bin_ctr_lr, ltg_lr)
        axs[3].plot(x_bin_ctr_lr, regr_lr_nonltg.intercept + regr_lr_nonltg.slope*x_bin_ctr_lr, color='k', linewidth=1, linestyle='dashdot', alpha=0.2)
        axs[3].plot(x_bin_ctr_lr, regr_lr_ltg.intercept + regr_lr_ltg.slope*x_bin_ctr_lr, color='tab:olive', linewidth=1, linestyle='dashdot', alpha=0.2)
    histo_fracs = histo_fracs[:, invalid_histo_mask]
    x_bin_ctr = x_bin_ctr[invalid_histo_mask]
    [axs[2].scatter(x_bin_ctr, histo_fracs[i], color=styles_dict[key]['color'], s=2) for i, key in enumerate(things_to_plot.keys())]
    regrs = [linregress(x_bin_ctr, histo_fracs[i]) for i in range(histo_fracs.shape[0])]
    if should_xlog:
        regrs_log = [linregress(np.log10(x_bin_ctr), histo_fracs[i]) for i in range(histo_fracs.shape[0])]
        regrs_pval_str = [f'={regrs_log[i].pvalue:.3f}' if regrs_log[i].pvalue >= 0.001 else f'<.001' for i in range(len(regrs_log))]
        regrs_labels = [f'm={regrs_log[i].slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'+
                        f'b={regrs_log[i].intercept:.0f}%\n'+r'r$^{2}$'+f'={regrs_log[i].rvalue**2:.2f}, p{regrs_pval_str[i]}' 
                        if not np.isnan(regrs_log[i].rvalue) else '(no tracks)\n' for i in range(len(regrs_log))]
    else:
        regrs_pval_str = [f'={regrs[i].pvalue:.3f}' if regrs[i].pvalue >= 0.001 else f'<.001' for i in range(len(regrs))]
        regrs_labels = [f'm={regrs[i].slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'+
                        f'b={regrs[i].intercept:.0f}%\n'+r'r$^{2}$'+f'={regrs[i].rvalue**2:.2f}, p{regrs_pval_str[i]}' 
                        if not np.isnan(regrs[i].rvalue) else '(no tracks)\n' for i in range(len(regrs))]
    reg_handles = [axs[2].plot(x_bin_ctr, regrs[i].intercept + regrs[i].slope*x_bin_ctr, color=styles_dict[key]['color'], linewidth=1, linestyle='--', label=regrs_labels[i]) for i, key in enumerate(things_to_plot.keys())]
    axs[2].set_ylabel('Percentage of Tracks')
    nonltg = np.zeros(histo_fracs.shape[1], dtype=float)
    ltg = np.zeros(histo_fracs.shape[1], dtype=float)
    for i, histo_frac in enumerate(histo_fracs):
        if 'Lightning' in styles_dict[list(things_to_plot.keys())[i]]['label']:
            ltg += histo_frac
        else:
            nonltg += histo_frac
    invalid_mask = (nonltg + ltg) == 0
    x_bin_ctr = x_bin_ctr[~invalid_mask]
    nonltg = nonltg[~invalid_mask]
    ltg = ltg[~invalid_mask]
    nonltg_pts = axs[3].scatter(x_bin_ctr, nonltg, color='k', s=2, label='All non-lightning')
    ltg_pts = axs[3].scatter(x_bin_ctr, ltg, color='tab:olive', s=2, label='All lightning')

    nonltg_regr = linregress(x_bin_ctr, nonltg)
    ltg_regr = linregress(x_bin_ctr, ltg)
    if should_xlog:
        regr_log_nonltg = linregress(np.log10(x_bin_ctr), nonltg)
        regr_log_ltg = linregress(np.log10(x_bin_ctr), ltg)
        regr_label_nonltg = f'm={regr_log_nonltg.slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'
        nonltg_pval_str = f'={regr_log_nonltg.pvalue:.3f}' if regr_log_nonltg.pvalue >= 0.001 else f'<.001'
        regr_label_nonltg += f'b={regr_log_nonltg.intercept:.0f}%\n'+r'r$^{2}$'+f'={regr_log_nonltg.rvalue**2:.2f}, p{nonltg_pval_str}'
        regr_label_ltg = f'm={regr_log_ltg.slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'
        ltg_pval_str = f'={regr_log_ltg.pvalue:.3f}' if regr_log_ltg.pvalue >= 0.001 else f'<.001'
        regr_label_ltg += f'b={regr_log_ltg.intercept:.0f}%\n'+r'r$^{2}$'+f'={regr_log_ltg.rvalue**2:.2f}, p{ltg_pval_str}'
    else:
        regr_label_nonltg = f'm={nonltg_regr.slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'
        nonltg_pval_str = f'={nonltg_regr.pvalue:.3f}' if nonltg_regr.pvalue >= 0.001 else f'<.001'
        regr_label_nonltg += f'b={nonltg_regr.intercept:.0f}%\n'+r'r$^{2}$'+f'={nonltg_regr.rvalue**2:.2f}, p{nonltg_pval_str}'
        regr_label_ltg = f'm={ltg_regr.slope:.2e} %/{all_tracks[dv].attrs.get('units', '')}\n'
        ltg_pval_str = f'={ltg_regr.pvalue:.3f}' if ltg_regr.pvalue >= 0.001 else f'<.001'
        regr_label_ltg += f'b={ltg_regr.intercept:.0f}%\n'+r'r$^{2}$'+f'={ltg_regr.rvalue**2:.2f}, p{ltg_pval_str}'
    nonltg_regr_handles = axs[3].plot(x_bin_ctr, nonltg_regr.intercept + nonltg_regr.slope*x_bin_ctr, color='k', linewidth=1, linestyle='dashdot', label=regr_label_nonltg)
    ltg_regr_handles = axs[3].plot(x_bin_ctr, ltg_regr.intercept + ltg_regr.slope*x_bin_ctr, color='tab:olive', linewidth=1, linestyle='dashdot', label=regr_label_ltg)

    [ax.set_xlim(x_bin0, x_binf) for ax in axs[:4]]

    axs[0].set_ylabel('Track Count')
    [ax.set_ylabel('Track Percentage') for ax in axs[2:4]]
    [ax.set_ylim(0, 100) for ax in axs[1:4]]
    axs[3].set_xlabel(f'{all_tracks[dv].attrs.get('long_name', dv).replace('brightness', 'brightness\n').replace('summed, averaged',
                    'summed,\naveraged').replace('km in fea', 'km\n in fea')}{fancy_string} ({all_tracks[dv].attrs.get('units', '')})')
    if should_xlog:
        for ax in axs:
            ax.set_xscale('log')
            axs[0].set_yscale('log')
    if dv == 'near_seabreeze_time':
        track_count_ax.set_yscale('log')
    [ax.xaxis.set_ticklabels([]) for ax in axs[0:2]]
    cat_handles = hist_handles
    cat_handles.extend([nonltg_pts, ltg_pts])
    reg_handles = [r[0] for r in reg_handles]

    lax1.legend(handles=[ellipse, all_reg_handle[0], ellipse_ltg, ltg_reg_handle[0], nonltg_regr_handles[0], ltg_regr_handles[0]], loc='center', ncols=4)
    axs[4].legend(handles=cat_handles, labels=[cat_handles[i].get_label()+'\n'+reg_handles[i].get_label() for i in range(len(reg_handles))], loc='center', ncols=4)
    axs[4].axis('off')
    fig.tight_layout()
    cbar_ax.set_position([0.02, 0.93, 0.285, 0.01])
    ax.set_position([0.065, 0.4, 0.24, 0.5])
    ltg_reg_ax.set_position([0.38, 0.46, 0.26, 0.26])
    track_count_ax.set_position([0.38, 0.73, 0.26, 0.26])
    cat_reg_ax.set_position([0.7, 0.46, 0.26, 0.26])
    track_percent_ax.set_position([0.7, 0.73, 0.26, 0.26])
    lax1.set_position([0, 0.21, 1, 0.11])
    legend_ax.set_position([0, 0.045, 1, 0.11])
    return fig

In [ ]:
make_combined_frequency_presence('track_mlecape', 'mean', np.arange(0, 3500, 50)).savefig('mwr_figs/mlecape.pdf') # Fig. 3
make_combined_frequency_presence('track_ccn_profile_0.6', 'mean', np.arange(0, 5001, 250), should_xlog=False, isel={'vertical_levels' : 0}).savefig('mwr_figs/ccn0.6.pdf') # Fig. 5
make_combined_frequency_presence('track_zdrvol', 'sum', np.logspace(0, 3, 50), should_xlog=True).savefig('mwr_figs/zdrvol.pdf') # Fig. 7
make_combined_frequency_presence('track_kdpvol', 'sum', np.logspace(0, 2.5, 16), should_xlog=True).savefig('mwr_figs/kdpvol.pdf') # Fig. 7
make_combined_frequency_presence('track_seabreeze_proximity', 'min', np.arange(0, 121, 5)).savefig('mwr_figs/seabreeze_proximity.pdf') # Fig. 10
make_combined_frequency_presence('near_seabreeze_time', 'nothing', np.arange(0, 121, 5)).savefig('mwr_figs/near_seabreeze_time.pdf') # Fig. 10
plt.close('all')

In [ ]:
def make_bivariate(xdv, this_func_x, x_bins, ydv, this_func_y, y_bins, isel_x=None, isel_y=None):
    ltg_mask = all_tracks.track_flash_count.sum(dim='timestep') > 0
    thing_to_plot_x = all_tracks[xdv]
    cats_to_plot_x = {
        'nothing' : nothing_tracks[xdv],
        'zdr' : zdr_tracks[xdv],
        'kdp' : kdp_tracks[xdv],
        'zdr_kdp' : zdr_kdp_tracks[xdv],
        'zdr_ltg' : zdr_ltg_tracks[xdv],
        'kdp_ltg' : kdp_ltg_tracks[xdv],
        'zdr_kdp_ltg' : zdr_kdp_ltg_tracks[xdv],
        'ltg' : ltg_tracks[xdv]
    }
    if isel_x is not None:
        thing_to_plot_x = thing_to_plot_x.isel(**isel_x)
        for key, val in cats_to_plot_x.items():
            cats_to_plot_x[key] = val.isel(**isel_x)
    if this_func_x == 'mean':
        thing_to_plot_x = thing_to_plot_x.mean(dim='timestep', skipna=True)
        for key, val in cats_to_plot_x.items():
            cats_to_plot_x[key] = val.mean(dim='timestep', skipna=True)
        fancy_string_x = ',\nmean along track'
    elif this_func_x == 'min':
        thing_to_plot_x = thing_to_plot_x.min(dim='timestep', skipna=True)
        for key, val in cats_to_plot_x.items():
            cats_to_plot_x[key] = val.min(dim='timestep', skipna=True)
        fancy_string_x = ',\nminimum along track'
    elif this_func_x == 'max':
        thing_to_plot_x = thing_to_plot_x.max(dim='timestep', skipna=True)
        for key, val in cats_to_plot_x.items():
            cats_to_plot_x[key] = val.max(dim='timestep', skipna=True)
        fancy_string_x = ',\nmaximum along track'
    elif this_func_x == 'sum':
        thing_to_plot_x = thing_to_plot_x.sum(dim='timestep', skipna=True)
        for key, val in cats_to_plot_x.items():
            cats_to_plot_x[key] = val.sum(dim='timestep', skipna=True)
        fancy_string_x = ',\nsum along track'
    elif this_func_x == 'nothing':
        fancy_string_x = ''
    else:
        raise ValueError(f"Unknown function {this_func_x} for variable {xdv}")
    if 'time' in thing_to_plot_x.name:
        thing_to_plot_x = thing_to_plot_x.data.astype(float)
    
    thing_to_plot_y = all_tracks[ydv]
    cats_to_plot_y = {
        'nothing' : nothing_tracks[ydv],
        'zdr' : zdr_tracks[ydv],
        'kdp' : kdp_tracks[ydv],
        'zdr_kdp' : zdr_kdp_tracks[ydv],
        'zdr_ltg' : zdr_ltg_tracks[ydv],
        'kdp_ltg' : kdp_ltg_tracks[ydv],
        'zdr_kdp_ltg' : zdr_kdp_ltg_tracks[ydv],
        'ltg' : ltg_tracks[ydv]
    }
    if isel_y is not None:
        thing_to_plot_y = thing_to_plot_y.isel(**isel_y)
        for key, val in cats_to_plot_y.items():
            cats_to_plot_y[key] = val.isel(**isel_y)
    if this_func_y == 'mean':
        cats_to_plot_y = {key: val.mean(dim='timestep') for key, val in cats_to_plot_y.items()}
        thing_to_plot_y = thing_to_plot_y.mean(dim='timestep')
        fancy_string = ',\nmean along track'
    elif this_func_y == 'min':
        cats_to_plot_y = {key: val.min(dim='timestep') for key, val in cats_to_plot_y.items()}
        thing_to_plot_y = thing_to_plot_y.min(dim='timestep')
        fancy_string = ',\nmin along track'
    elif this_func_y == 'max':
        cats_to_plot_y = {key: val.max(dim='timestep') for key, val in cats_to_plot_y.items()}
        thing_to_plot_y = thing_to_plot_y.max(dim='timestep')
        fancy_string = ',\nmax along track'
    elif this_func_y == 'sum':
        cats_to_plot_y = {key: val.sum(dim='timestep') for key, val in cats_to_plot_y.items()}
        thing_to_plot_y = thing_to_plot_y.sum(dim='timestep')
        fancy_string = ',\nsum along track'
    elif this_func_y == 'nothing':
        fancy_string = ''
    else:
        raise ValueError(f"Unknown function {this_func_y} for variable {ydv}")
    if 'time' in thing_to_plot_y.name:
        thing_to_plot_y = thing_to_plot_y.astype(float)
    x_bin0 = x_bins[0]
    x_binf = x_bins[-1]
    x_bins[0] = np.min([thing_to_plot_x.min(), x_bin0])
    x_bins[-1] = np.max([thing_to_plot_x.max(), x_binf])

    y_bin0 = y_bins[0]
    y_binf = y_bins[-1]
    y_bins[0] = np.min([thing_to_plot_y.min(), y_bin0])
    y_bins[-1] = np.max([thing_to_plot_y.max(), y_binf])

    fig = plt.figure(figsize=(6.5, 6.5))
    gs = GridSpec(3, 2, figure=fig, height_ratios=[1e-4, 1, 1])
    lax = fig.add_subplot(gs[0, :])
    axs = [fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]), fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[2, 1])]
    count_hist_art = axs[0].hist2d(
        thing_to_plot_x.data,
        thing_to_plot_y.data,
        cmap='viridis', norm=pltcolors.LogNorm(), bins=[x_bins, y_bins], range=[[thing_to_plot_x.min(), thing_to_plot_x.max()], [thing_to_plot_y.min(), thing_to_plot_y.max()]], rasterized=True
    )
    axs[0].set_xlim(x_bin0, x_binf)
    axs[0].set_ylim(y_bin0, y_binf)
    axs[0].set_title('Track Distribution')
    fig.colorbar(count_hist_art[3], ax=axs[0], label='Track Count', orientation='vertical')
    cat_handles = []
    for i, key in enumerate(cats_to_plot_y.keys()):
        if cats_to_plot_y[key].size == 0:
            this_cat_handle = axs[1].scatter([np.nan], [np.nan], color=styles_dict[key]['color'], label=styles_dict[key]['label'], s=15, marker='+')
            cat_handles.append(this_cat_handle)
            continue
        cat_to_plot_x = cats_to_plot_x[key].data
        cat_to_plot_y = cats_to_plot_y[key].data
        cat_to_plot_x_nonnan = cat_to_plot_x[~np.isnan(cat_to_plot_x) & ~np.isnan(cat_to_plot_y)]
        cat_to_plot_y_nonnan = cat_to_plot_y[~np.isnan(cat_to_plot_x) & ~np.isnan(cat_to_plot_y)]
        if cat_to_plot_x_nonnan.size <= 1:
            this_cat_handle = axs[1].scatter([np.nan], [np.nan], color=styles_dict[key]['color'], label=styles_dict[key]['label'], s=15, marker='+')
            cat_handles.append(this_cat_handle)
            continue
        mu_x, mu_y, ell_x, angle = calculate_error_ellipse(cat_to_plot_x_nonnan, cat_to_plot_y_nonnan)
        this_cat_handle = axs[1].scatter(mu_x, mu_y, color=styles_dict[key]['color'], label=styles_dict[key]['label'], s=15, marker='+')
        cat_handles.append(this_cat_handle)
        ellipse = Ellipse((mu_x, mu_y), width=ell_x[0], height=ell_x[1], angle=angle, color=styles_dict[key]['color'], label=styles_dict[key]['label'], fill=False)
        axs[1].add_patch(ellipse)
    axs[1].set_xlim(x_bin0, x_binf)
    axs[1].set_ylim(y_bin0, y_binf)
    axs[1].set_title('Track Category Error Ellipses')
    lax.legend(handles=cat_handles, ncols=4, loc='center')
    lax.axis('off')
    flash_hist_art = axs[2].hist2d(
        thing_to_plot_x.data,
        thing_to_plot_y.data,
        cmap='viridis', norm=pltcolors.LogNorm(), bins=[x_bins, y_bins], range=[[thing_to_plot_x.min(), thing_to_plot_x.max()], [thing_to_plot_y.min(), thing_to_plot_y.max()]], rasterized=True, weights=all_tracks.track_flash_count.sum(dim='timestep', skipna=True)
    )
    axs[2].set_xlim(x_bin0, x_binf)
    axs[2].set_ylim(y_bin0, y_binf)
    axs[2].set_title('Lightning Distribution')
    fig.colorbar(flash_hist_art[3], ax=axs[2], label='Track Flash Count', orientation='vertical')

    thing_to_plot_x_ltg = thing_to_plot_x.isel(track=ltg_mask)
    thing_to_plot_y_ltg = thing_to_plot_y.isel(track=ltg_mask)

    n_tracks_ltg = np.histogram2d(thing_to_plot_x_ltg.data, thing_to_plot_y_ltg.data, bins=[x_bins, y_bins], range=[[thing_to_plot_x.min(), thing_to_plot_x.max()], [thing_to_plot_y.min(), thing_to_plot_y.max()]])
    n_tracks_hist = np.histogram2d(thing_to_plot_x.data, thing_to_plot_y.data, bins=[x_bins, y_bins], range=[[thing_to_plot_x.min(), thing_to_plot_x.max()], [thing_to_plot_y.min(), thing_to_plot_y.max()]])
    fraction_ltg = n_tracks_ltg[0] / n_tracks_hist[0]
    frac_cmap = plt.get_cmap('viridis')
    frac_cmap.set_under('#A0A0A0')
    ltg_fraction_handle = axs[3].pcolormesh(x_bins, y_bins, fraction_ltg.T, cmap=frac_cmap, rasterized=True, vmin=1e-11, vmax=1)

    fig.colorbar(ltg_fraction_handle, ax=axs[3], label='Fraction of tracks with lightning', orientation='vertical', extend='min')
    axs[3].set_xlim(x_bin0, x_binf)
    axs[3].set_ylim(y_bin0, y_binf)
    axs[3].set_title('Lightning Fraction Distribution')
    ylabel_str = f'{all_tracks[ydv].attrs.get("long_name", ydv).replace("brightness", "brightness\n").replace("summed, averaged",
                "summed,\naveraged").replace("km in fea", "km\n in fea")}{fancy_string} ({all_tracks[ydv].attrs.get("units", "")})'
    fig.supxlabel(f'{all_tracks[xdv].attrs.get("long_name", xdv)}{fancy_string} ({all_tracks[xdv].attrs.get("units", "")})')
    fig.supylabel(ylabel_str)
    fig.tight_layout()
    return fig

In [ ]:
ccn_ecape_bivariate_fig = make_bivariate('track_ccn_profile_0.6', 'mean', np.arange(0, 5001, 250), 'track_mlecape', 'mean', np.arange(0, 3501, 50), isel_x={'vertical_levels' : 0})
ccn_ecape_bivariate_fig.savefig('mwr_figs/bivariate_ccn_ecape.pdf')
ecape_cinh_bivariate_fig = make_bivariate('track_mlecape', 'mean', np.arange(0, 3501, 50), 'track_mlcin', 'mean', np.arange(-250, 1, 5))
ecape_cinh_bivariate_fig.savefig('mwr_figs/bivariate_ecape_cinh.pdf')
ccn_echotop_bivariate_fig = make_bivariate('track_ccn_profile_0.6', 'mean', np.arange(0, 5001, 250), 'track_echotop', 'max', np.arange(0, 15.1, 0.5), isel_x={'vertical_levels' : 0})
ccn_echotop_bivariate_fig.savefig('mwr_figs/bivariate_ccn_echotop.pdf')
plt.close('all')

In [ ]:
cutoff = 30
things_to_plot = {
    'nothing' : nothing_normalized.track_present.sum(dim='track'),
    'zdr' : zdr_normalized.track_present.sum(dim='track'),
    'kdp' : kdp_normalized.track_present.sum(dim='track'),
    'zdr_kdp' : zdr_kdp_normalized.track_present.sum(dim='track'),
    'zdr_ltg' : zdr_ltg_normalized.track_present.sum(dim='track'),
    'kdp_ltg' : kdp_ltg_normalized.track_present.sum(dim='track'),
    'zdr_kdp_ltg' : zdr_kdp_ltg_normalized.track_present.sum(dim='track'),
    'ltg' : ltg_normalized.track_present.sum(dim='track')
}
normalized_times.track_present.sum(dim='track')
fig = plt.figure(figsize=(6.5, 5))
normalized_time_gs = GridSpec(4, 2, figure=fig, height_ratios=[0.1, 1, 1, 1], hspace=0.3)
contributing_categories_ax = fig.add_subplot(normalized_time_gs[1, 0])
contributing_categories_ax_zoom = fig.add_subplot(normalized_time_gs[1, 1])
for this_ax in [contributing_categories_ax, contributing_categories_ax_zoom]:
    handles = []
    total_plot = this_ax.plot(normalized_times.time/60, normalized_times.track_present.sum(dim='track'), color='k', linewidth=1.2, label='Total')
    for key in things_to_plot.keys():
        cat_plot = this_ax.plot(normalized_times.time/60, things_to_plot[key], label=styles_dict[key]['label'], color=styles_dict[key]['color'], linewidth=1)
        handles.append(cat_plot[0])
    handles.append(total_plot[0])
    this_ax.set_xlabel('Time since track start (minutes)')
    this_ax.axvline(cutoff, color='r', linestyle='--', label=f'{cutoff} min cutoff', linewidth=1)
contributing_categories_ax.set_yscale('log')
contributing_categories_ax.set_ylabel('Number of tracks')
contributing_categories_ax_zoom.set_xlim(0, cutoff)
legend_ax = fig.add_subplot(normalized_time_gs[0, :])
legend_ax.legend(handles=handles, loc='center', ncols=5)
legend_ax.axis('off')
zdr_strength_ax = fig.add_subplot(normalized_time_gs[2, :])
zdr_strength_ax_zoom = fig.add_subplot(normalized_time_gs[3, :])
zdr_zdr = zdr_normalized.track_zdrvol * (0.5)**3
zdr_mu = zdr_zdr.mean(dim='track')
zdr_iqr = zdr_zdr.quantile(0.75, dim='track') - zdr_zdr.quantile(0.25, dim='track')
everything_zdr = zdr_kdp_ltg_normalized.track_zdrvol * (0.5)**3
zdr_kdp_ltg_mu = everything_zdr.mean(dim='track')
zdr_kdp_ltg_iqr = everything_zdr.quantile(0.75, dim='track') - everything_zdr.quantile(0.25, dim='track')
zdr_strength_ax.set_xlabel('Time since track start (minutes)')
for this_ax in [zdr_strength_ax, zdr_strength_ax_zoom]:
    this_ax.plot(normalized_times.time/60, zdr_mu, color='tab:orange', label=r'Z$_{DR}$', linewidth=1)
    this_ax.plot(normalized_times.time/60, zdr_kdp_ltg_mu, color='tab:pink', label=r'Z$_{DR}$ K$_{DP}$ Lightning', linewidth=1)
    this_ax.fill_between(normalized_times.time/60, zdr_mu - zdr_iqr, zdr_mu + zdr_iqr, color='tab:orange', alpha=0.3)
    this_ax.fill_between(normalized_times.time/60, zdr_kdp_ltg_mu - zdr_kdp_ltg_iqr, zdr_kdp_ltg_mu + zdr_kdp_ltg_iqr, color='tab:pink', alpha=0.3,)
    this_ax.axhline(0, color='gray', linestyle='--', linewidth=0.2)
    this_ax.legend(loc='upper left')
zdr_strength_ax.set_ylabel(r'Mean Z$_{DR}$ column volume' + '\n' + r'(km$^3$)')
zdr_strength_ax.set_xlim(0, normalized_times.time.max()/60)
zdr_strength_ax_zoom.set_xlim(0, cutoff)

zdr_strength_ax.set_ylim(-0.01, (zdr_kdp_ltg_mu + zdr_kdp_ltg_iqr).max())
zdr_strength_ax_zoom.set_ylim(-0.01, (zdr_kdp_ltg_mu.isel(time=normalized_times.time/60 < cutoff) + zdr_kdp_ltg_iqr.isel(time=normalized_times.time/60 < cutoff)).max())
zdr_strength_ax_zoom.set_xlabel('Time since track start (minutes)')
fig.savefig('mwr_figs/along_track_zdrvol.pdf')
plt.close('all')

In [ ]:
khgx_lat, khgx_lon = 29.471900939941406, -95.0787353515625
IAH_lat, IAH_lon = 29.9844353, -95.3414425
HOU_lat, HOU_lon = 29.6457998, -95.2772316
# loud sounding launch sites
# Load the ARM DOE sondes
arm_sonde_files = sorted(glob('/Volumes/LtgSSD/arm-sondes/*.cdf'))
arm_sonde_lons = []
arm_sonde_lats = []

for sonde_file in arm_sonde_files:
    tmp_sonde = xr.open_dataset(sonde_file)
    arm_sonde_lons.append(tmp_sonde.lon.data[0])
    arm_sonde_lats.append(tmp_sonde.lat.data[0])
    tmp_sonde.close()
# Load the TAMU sondes
tamu_sonde_path = '/Volumes/LtgSSD/TAMU_SONDES/'
tamu_sonde_files = [s.replace(tamu_sonde_path, '') for s in sorted(glob('/Volumes/LtgSSD/TAMU_SONDES/*_TSPOTINT.txt'))]
tamu_sonde_files_split = np.vstack(np.char.split(tamu_sonde_files, sep='_'))
tamu_sonde_lons = tamu_sonde_files_split[:, -3]
lon_negative = ((np.char.find(tamu_sonde_lons, 'W') >= 0).astype(int) - 0.5) * -2
tamu_sonde_lons = np.char.replace(tamu_sonde_lons, 'W', '')
tamu_sonde_lons = np.char.replace(tamu_sonde_lons, 'E', '')
tamu_sonde_lons = tamu_sonde_lons.astype(float) * lon_negative

tamu_sonde_lats = tamu_sonde_files_split[:, -2]
lat_negative = ((np.char.find(tamu_sonde_lats, 'S') >= 0).astype(int) - 0.5) * -2
tamu_sonde_lats = np.char.replace(tamu_sonde_lats, 'S', '')
tamu_sonde_lats = np.char.replace(tamu_sonde_lats, 'N', '')
tamu_sonde_lats = tamu_sonde_lats.astype(float) * lat_negative

# Load the CMAS sondes
CMAS_sonde_path = '/Volumes/LtgSSD/CMAS-sondes/'
CMAS_sonde_files = sorted(glob('/Volumes/LtgSSD/CMAS-sondes/GrawSonde*.nc'))
CMAS_sonde_lons = []
CMAS_sonde_lats = []

for sonde_file in CMAS_sonde_files:
    tmp_sonde = xr.open_dataset(sonde_file)
    CMAS_sonde_lons.append(tmp_sonde.longitude.data[0])
    CMAS_sonde_lats.append(tmp_sonde.latitude.data[0])
    tmp_sonde.close()

CMAS_sonde_lons = np.array(CMAS_sonde_lons)
CMAS_sonde_lats = np.array(CMAS_sonde_lats)

arm_latlons = np.array([arm_sonde_lats, arm_sonde_lons])
unique_arm_latlons = np.unique(arm_latlons, axis=1)
tamu_latlons = np.array([tamu_sonde_lats, tamu_sonde_lons])
unique_tamu_latlons = np.unique(tamu_latlons, axis=1)
CMAS_latlons = np.array([CMAS_sonde_lats, CMAS_sonde_lons])
unique_CMAS_latlons = np.unique(np.round(CMAS_latlons, 2), axis=1)

kghx = gv.Points((khgx_lon, khgx_lat), kdims=['Longitude', 'Latitude'], label='KHGX Radar').opts(color='gold', line_color='black', marker='star', size=15, tools=['hover'], width=1000, height=1000, toolbar=None)
arm_laporte = gv.Points((unique_arm_latlons[1, 1], unique_arm_latlons[0, 1]), kdims=['Longitude', 'Latitude'], label='ARM M1 (La Porte, TX)').opts(color='green', size=15, marker='star', line_color='black', tools=['hover'], width=1000, height=1000)
arm_damon = gv.Points((unique_arm_latlons[1, 0], unique_arm_latlons[0, 0]), kdims=['Longitude', 'Latitude'], label='ARM S3 (Damon, TX)').opts(color='red', size=15, marker='star', line_color='black', tools=['hover'], width=1000, height=1000)
tamu_sondes = gv.Points((unique_tamu_latlons[1, :], unique_tamu_latlons[0, :]), kdims=['Longitude', 'Latitude'], label='Texas A&M Mobile Deployment Sites').opts(color='maroon', size=10, tools=['hover'], width=1000, height=1000)
cmas_sondes = gv.Points((unique_CMAS_latlons[1, :], unique_CMAS_latlons[0, :]), kdims=['Longitude', 'Latitude'], label='Center for Multiscale Applied Sensing Mobile Deployment Sites').opts(color='blue', size=10, tools=['hover'])
hou_airport = gv.Points((HOU_lon, HOU_lat), kdims=['Longitude', 'Latitude'], label='HOU Airport').opts(color='black', size=15, marker='triangle', line_color='black', tools=['hover'], width=1000, height=1000)
iah_airport = gv.Points((IAH_lon, IAH_lat), kdims=['Longitude', 'Latitude'], label='IAH Airport').opts(color='black', size=15, marker='square', line_color='black', tools=['hover'], width=1000, height=1000)
my_dash = kghx * arm_laporte * arm_damon * tamu_sondes * cmas_sondes * iah_airport * hou_airport * gv.tile_sources.OSM

hv.save(my_dash, './mwr_figs/assets.png') # Figure 1